1. Impor Pustaka dan Memuat Dataset

Melakukan impor pustaka utama (pandas, numpy, matplotlib, seaborn) beserta modul dari scikit-learn untuk pemodelan regresi. Setelah itu, membaca file dataset yugioh_cards.csv serta menampilkan informasi dasar dan jumlah baris dari dataset tersebut.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv('yugioh_cards.csv')

print("=== Informasi Dataset Awal ===")
print(f"Total baris: {df.shape[0]}")
print("\n=== 10 Baris Pertama Dataset ===")
display(df.head(10))

2. Data Preprocessing dan Exploratory Data Analysis (EDA)

Membersihkan data dengan membuang kartu yang tidak memiliki atribut atk, def, dan level (seperti Spell atau Trap Card) karena kita hanya akan menganalisis kartu Monster. Nilai harga (tcgplayer_price) yang kosong diisi dengan 0. Kemudian, menampilkan grafik distribusi harga kartu di bawah 100 Dolar untuk melihat sebaran data secara visual.

In [ ]:
df_clean = df.dropna(subset=['atk', 'def', 'level']).copy()
df_clean['tcgplayer_price'] = df_clean['tcgplayer_price'].fillna(0)

print(f"Total data khusus Monster Card yang siap diolah: {df_clean.shape[0]}\n")

plt.figure(figsize=(10, 5))
sns.histplot(df_clean[df_clean['tcgplayer_price'] < 100]['tcgplayer_price'], bins=50, kde=True, color='purple')
plt.title('Distribusi Harga Kartu Yu-Gi-Oh (Di bawah $100)')
plt.xlabel('Harga (Dolar)')
plt.ylabel('Frekuensi (Jumlah Kartu)')
plt.show()

3. Penentuan Fitur dan Data Splitting

Menentukan variabel independen atau fitur (X) yang terdiri dari level, atk, dan def, serta variabel dependen atau target (y) yaitu tcgplayer_price. Data kemudian dipisah menjadi data latih (training set) sebesar 80 persen dan data uji (testing set) sebesar 20 persen.

In [ ]:
fitur_x = ['level', 'atk', 'def']
target_y = 'tcgplayer_price'

X = df_clean[fitur_x]
y = df_clean[target_y]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Jumlah data training: {X_train.shape[0]}")
print(f"Jumlah data testing: {X_test.shape[0]}")

4. Modeling dan Evaluasi

Membangun model regresi menggunakan algoritma RandomForestRegressor. Model dilatih menggunakan data latih, kemudian diuji untuk memprediksi harga pada data uji. Performa model dievaluasi menggunakan metrik Mean Absolute Error (MAE), Root Mean Squared Error (RMSE), dan R-Squared (R2 Score).

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("=== HASIL EVALUASI MODEL ===")
print(f"Mean Absolute Error (MAE): ${mae:.2f}")
print(f"Root Mean Squared Error (RMSE): ${rmse:.2f}")
print(f"R-Squared (R2 Score): {r2:.4f}")

5. Visualisasi Hasil Prediksi dan Feature Importance

Menampilkan dua visualisasi akhir:
1. Scatter Plot untuk membandingkan harga asli (aktual) dengan tebakan model (prediksi).
2. Bar Chart tingkat kepentingan fitur (Feature Importance) untuk melihat variabel fisik kartu mana yang paling memengaruhi harga pasar.

In [ ]:
# 1. Visualisasi Prediksi vs Aktual
plt.figure(figsize=(10, 6))
plt.scatter(y_test, y_pred, alpha=0.5, color='blue', edgecolors='k')
harga_maksimal = max(y_test.max(), y_pred.max())
plt.plot([0, harga_maksimal], [0, harga_maksimal], 'r--', lw=2) 
plt.xlabel("Harga Asli ($) - Aktual")
plt.ylabel("Tebakan Harga Model ($) - Prediksi")
plt.title("Perbandingan Harga Asli vs Prediksi Model")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()


In [ ]:
# 2. Visualisasi Feature Importance
plt.figure(figsize=(8, 5))
importances = model.feature_importances_
sns.barplot(x=fitur_x, y=importances, hue=fitur_x, palette='viridis', legend=False)
plt.title('Fitur Apa yang Paling Memengaruhi Harga Kartu?')
plt.xlabel('Fitur (Spesifikasi Kartu)')
plt.ylabel('Tingkat Kepentingan (Skala 0 - 1)')
plt.show()